# Part C: Mage Iris pipeline (Colab runner)
Works inside your cloned repo, so everything you do is committed from here. Run top to bottom.

In [ ]:
import os, subprocess, sys, getpass
GITHUB_USER, REPO = "sashankdadi", "freshcart-pipelines"
def sh(cmd, check=False):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = r.stdout + r.stderr
    print(out); return out

## 1. Install Mage in a Python 3.12 environment (as in class)

In [ ]:
sh("pip install -q uv && uv venv -q -p 3.12 /content/mage_env && uv pip install -q --python /content/mage_env mage-ai scikit-learn pytest")
os.environ["PATH"] = "/content/mage_env/bin:" + os.environ["PATH"]
sh("mage --help | head -n 5")

## 2. Clone your repo and add Mage's own scaffold files (without overwriting yours)

In [ ]:
if not os.path.exists(REPO):
    sh(f"git clone https://github.com/{GITHUB_USER}/{REPO}.git")
os.chdir(REPO)
sh("mage init _scaffold && cp -rn _scaffold/. iris_project/ && rm -rf _scaffold && ls iris_project")

## 3. A3: run the unit tests (same thing CI runs)

In [ ]:
sh("/content/mage_env/bin/python -m pytest -v")

## 4. C1: run the whole pipeline with its tests

In [ ]:
sh("mage run iris_project iris_classifier --test")

## 5. C2: the test gate. For each max_depth delete the CSV, run, record accuracy / test result / file

In [ ]:
import re, pandas as pd
rows = []
for depth in [1, 2, 3, 4, 5]:
    if os.path.exists("iris_predictions.csv"): os.remove("iris_predictions.csv")
    out = sh(f"mage run iris_project iris_classifier --test --runtime-vars '{{\"max_depth\": {depth}}}'")
    m = re.search(r"Test accuracy = ([0-9.]+)", out)
    failed = ("Model accuracy is below 80%" in out) or ("Traceback" in out and "test_accuracy" in out)
    rows.append({"max_depth": depth, "test_accuracy": float(m.group(1)) if m else None,
                 "test_accuracy passed?": "NO" if failed else "YES",
                 "iris_predictions.csv created?": "yes" if os.path.exists("iris_predictions.csv") else "no"})
pd.DataFrame(rows)

## 6. C3: the evaluation branch (`evaluate_model` is already in the repo; here you inspect and run it)

In [ ]:
print(open("iris_project/pipelines/iris_classifier/metadata.yaml").read())
sh("mage run iris_project iris_classifier --test")   # prints the confusion matrix and the metrics table

## 7. Mage UI through a Cloudflare tunnel (screenshot the graph with the branch)

In [ ]:
import time, urllib.request
mage_server = subprocess.Popen(["/content/mage_env/bin/mage", "start", "iris_project"],
                               stdout=open("mage.log", "w"), stderr=subprocess.STDOUT)
for i in range(60):
    try:
        urllib.request.urlopen("http://localhost:6789"); print("Mage is running"); break
    except Exception: time.sleep(2)
else: print("Mage did not start - check mage.log")

In [ ]:
sh("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared")
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:6789"],
                          stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
time.sleep(10)
link = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
print("Open Mage here:", link[0] if link else "Not ready yet - run this cell again")

## 8. C4: your own block
`validate_data` (no missing values + exactly 3 classes) is already wired between `load_iris` and `prepare_data`, with its own `@test`. Its unit test is in `tests/test_validate_data.py`. If you prefer another block from the assignment (metrics CSV exporter, KNN branch), add it the same way and register it in `metadata.yaml`.

## 9. Commit and push from Colab
Create a GitHub **personal access token** (repo scope) and paste it when asked. Never save it in a file.

In [ ]:
token = getpass.getpass("GitHub token: ")
sh('git config user.name "Sashank Dadi" && git config user.email "you@example.com"')
sh(f"git remote set-url origin https://{token}@github.com/{GITHUB_USER}/{REPO}.git")
sh('git add iris_project tests && git commit -m "Mage: evaluation branch and validate_data block" && git push')